In [1]:
import json

from pymongo import MongoClient

# Load data from JSONL file
client = MongoClient("mongodb://localhost:27017/")
db = client["ddxplus"]
collection_train = db["train-semistructured"]
collection_val = db["validate-semistructured"]
collection_test = db["test-semistructured"]

train_data = list(
    collection_train.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)
val_data = list(
    collection_val.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)
test_data = list(
    collection_test.find(
        {},
        {
            "_id": 0,
            "PATHOLOGY": 0,
            "EVIDENCES": 0,
            "EVIDENCES_JSON_V2": 0,
            "DIFFERENTIAL_DIAGNOSIS": 0,
        },
    )
)

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# show one train sample
print("Example train sample:")
print(json.dumps(train_data[0], indent=2))

TARGET_KEY = "DIFFERENTIAL_DIAGNOSIS_NOPROB"

Train samples: 1025602
Validation samples: 132448
Test samples: 134529
Example train sample:
{
  "AGE": 18,
  "SEX": "M",
  "INITIAL_EVIDENCE": "E_91",
  "EVIDENCES_JSON_V1": {
    "E_48": [],
    "E_50": [],
    "E_53": [],
    "E_54": [
      "V_161",
      "V_183"
    ],
    "E_55": [
      "V_89",
      "V_108",
      "V_167"
    ],
    "E_56": [
      "4"
    ],
    "E_57": [
      "V_123"
    ],
    "E_58": [
      "3"
    ],
    "E_59": [
      "3"
    ],
    "E_77": [],
    "E_79": [],
    "E_91": [],
    "E_97": [],
    "E_201": [],
    "E_204": [
      "V_10"
    ],
    "E_222": []
  },
  "DIFFERENTIAL_DIAGNOSIS_NOPROB": [
    "Bronchitis",
    "Pneumonia",
    "URTI",
    "Bronchiectasis",
    "Tuberculosis",
    "Influenza",
    "HIV (initial infection)",
    "Chagas"
  ]
}


In [2]:
import random

import torch

from origami import DataConfig, ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.training import TableLogCallback, array_f1, array_jaccard

# For reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

config = OrigamiConfig(
    data=DataConfig(
        numeric_mode="none",
    ),
    model=ModelConfig(
        d_model=256,
        n_layers=6,
        n_heads=8,
    ),
    training=TrainingConfig(
        num_epochs=1,
        learning_rate=0.001,
        eval_strategy="steps",
        eval_steps=100,
        eval_sample_size=100,
        eval_metrics={"jaccard": array_jaccard, "f1": array_f1},
        shuffle_keys=True,
        upscale_factor=4,
        batch_size=100,
        target_key=TARGET_KEY,
    ),
)

pipeline = OrigamiPipeline(config)
callback = TableLogCallback(print_every=10)

In [3]:
pipeline.fit(train_data, eval_data=val_data, epochs=50, callbacks=[callback], verbose=True)

Vocabulary size: 668
Model parameters: 3,571,356
Training device: mps
| step: 10 | epoch: 0 | lr: 1.00e-05 | batch_dt:  286ms | loss: 5.6318 |
| step: 20 | epoch: 0 | lr: 2.00e-05 | batch_dt:  304ms | loss: 5.3579 |
| step: 30 | epoch: 0 | lr: 3.00e-05 | batch_dt:  276ms | loss: 4.8954 |
| step: 40 | epoch: 0 | lr: 4.00e-05 | batch_dt:  302ms | loss: 4.3820 |
| step: 50 | epoch: 0 | lr: 5.00e-05 | batch_dt:  284ms | loss: 3.9646 |
| step: 60 | epoch: 0 | lr: 6.00e-05 | batch_dt:  311ms | loss: 3.6416 |
| step: 70 | epoch: 0 | lr: 7.00e-05 | batch_dt:  287ms | loss: 3.3900 |
| step: 80 | epoch: 0 | lr: 8.00e-05 | batch_dt:  268ms | loss: 3.1849 |
| step: 90 | epoch: 0 | lr: 9.00e-05 | batch_dt:  259ms | loss: 2.8982 |
| step: 100 | epoch: 0 | lr: 1.00e-04 | batch_dt:  305ms | loss: 2.6732 | val_f1: 0.0000 | val_jaccard: 0.0000 | val_loss: 2.6077 |
| step: 110 | epoch: 0 | lr: 1.10e-04 | batch_dt:  288ms | loss: 2.5046 |
| step: 120 | epoch: 0 | lr: 1.20e-04 | batch_dt:  255ms | loss: 2.

OrigamiPipeline(numeric_mode='none', fitted)

In [4]:
pipeline.save("ddxplus_origami_pipeline.pt")

In [ ]:
from origami import OrigamiPipeline

pipeline = OrigamiPipeline.load("ddxplus_origami_pipeline.pt")

In [5]:
from origami.training import array_f1, array_precision, array_recall

pipeline.evaluate(
    test_data,
    metrics={"f1": array_f1, "precision": array_precision, "recall": array_recall},
    batch_size=256,
    verbose=True,
)

Computing loss:   0%|          | 0/526 [00:00<?, ?it/s]

Predicting:   0%|          | 0/526 [00:00<?, ?it/s]

{'loss': 0.80065522425075,
 'f1': 0.9696027033936221,
 'precision': 0.9693926669285932,
 'recall': 0.9769237673150009}